# Phase 5: Final Evaluation (Champion Model with Selected Features)
**Client:** Crédit Nationale Azur
**Objective:** Train and evaluate the final, hyperparameter-tuned Random Forest classifier using the selected-features pipeline (k=7) to generate the definitive business metrics for deployment.

In this notebook, we recreate the strict data preparation pipeline (load → split → encode → select features → standardise) and then apply the mathematically optimised Random Forest parameters discovered during tuning.

In [1]:
# Required Base Imports for Phase 5
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2

## 1. The Strict Data Pipeline
We execute our required chronology: loading, splitting, encoding, selecting features, and standardising.

In [2]:
# 1. Load and Map Target
df = joblib.load('../data/cleaned_data.pkl')
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()

# 3. Encode Categorical Variables
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']
for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)
    
    dummies_test = pd.get_dummies(X_test[col], prefix=col, dtype=int)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 4. Feature Selection
selector = SelectKBest(score_func=chi2, k=7)
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)
selected_features = X_train.columns[selector.get_support()]
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

# 5. Standardisation
continuous_features = ['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend']
cols_to_scale = [col for col in continuous_features if col in selected_features]

for col in cols_to_scale:
    scaler = StandardScaler()
    scaler.fit(X_train_selected[col].values.reshape(-1, 1))
    X_train_selected[col] = scaler.transform(X_train_selected[col].values.reshape(-1, 1)).flatten()
    X_test_selected[col] = scaler.transform(X_test_selected[col].values.reshape(-1, 1)).flatten()

print("Data pipeline executed successfully. Ready for final modelling.")

Data pipeline executed successfully. Ready for final modelling.


## 2. Train Champion Model and Evaluate
We instantiate the Random Forest using the parameters discovered during Phase 4 tuning.

In [3]:
# Instantiate the tuned Random Forest model
final_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    criterion='gini', 
    random_state=42
)

# Fit the final model
final_model.fit(X_train_selected, y_train)

# Generate predictions on the test set
y_pred = final_model.predict(X_test_selected)

# Flatten the confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Calculate specific metrics for imbalanced classes
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Output the official evaluation metrics
print("Official Champion Model Evaluation (Tuned Random Forest)")
print("*" * 56)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print("*" * 56)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Official Champion Model Evaluation (Tuned Random Forest)
********************************************************
True Negatives (TN):  966
False Positives (FP): 54
False Negatives (FN): 57
True Positives (TP):  123
********************************************************
Precision: 0.6949
Recall:    0.6833
F1-Score:  0.6891
